In [0]:
dbutils.widgets.removeAll()

In [0]:
dbutils.secrets.listScopes()
dbutils.secrets.list(scope="accessScopeforADLS")
connectionString = dbutils.secrets.get(scope="accessScopeforADLS", key="storageAccessKey")

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
dbutils.widgets.text("container", "raw")
dbutils.widgets.text("catalogo", "catalog_dev")
dbutils.widgets.text("esquema", "bronze")
dbutils.widgets.text("storageName", "adlscontenedor")

In [0]:

container = dbutils.widgets.get("container")
catalogo = dbutils.widgets.get("catalogo")
esquema = dbutils.widgets.get("esquema")
storageName = dbutils.widgets.get("storageName")

In [0]:
ruta = f"abfss://{container}@{storageName}.dfs.core.windows.net/"
ventas = ruta + "orders.csv"

In [0]:
ventas_schema = StructType(fields=[StructField("ID_ORDER", StringType(), False),
                                  StructField("STORE", StringType(), False),
                                  StructField("REG", IntegerType(), True),
                                  StructField("DATE", DateType(), False),
                                  StructField("ID_ITEM", StringType(), False),
                                  StructField("QTY", IntegerType(), False),
                                  StructField("PRICE", DecimalType(12,4), False),
                                  StructField("DISCOUNT", DecimalType(12,4), False),
                                  StructField("ID_SELLER", StringType(), False),
                                  StructField("STATUS", StringType(), False)
                                  ])

In [0]:
%python
df_ventas = (spark.read.format("csv")
    .option("header", "true")
    .option("dateFormat", "dd/MM/yyyy")
    .schema(ventas_schema)
    .load(ventas))
display(df_ventas)

In [0]:
df_ventas_ingestion_date = df_ventas.withColumn("INGESTION_DATE", current_timestamp())

In [0]:
df_ventas_final = df_ventas_ingestion_date.select(
                                    "ID_ORDER",
                                    "STORE",
                                    "REG",
                                    "DATE",
                                    "ID_ITEM",
                                    "QTY",
                                    "PRICE",
                                    "DISCOUNT",
                                    "ID_SELLER",
                                    "STATUS",
                                    "INGESTION_DATE"
                                )


In [0]:
df_ventas_final.write.mode("overwrite").option("overwriteSchema", "true").insertInto(f"{catalogo}.{esquema}.ventas")

In [0]:
#%sql    
#select count(*) from catalog_dev.bronze.ventas limit 50;


/*spark.conf.set(f"fs.azure.account.key.{storageName}.dfs.core.windows.net", connectionString.strip())

df_ventas = spark.read.option('header', True)\
                           .option('inferSchema', True)\
                           .csv("abfss://raw-retail@adlsproject.dfs.core.windows.net/orders.csv")

display(df_ventas)